In [0]:
data = [(1,"bharath"),(2,"arjun"),(3,"raj"),(4,"sai"),(1,"bharath")]

df = spark.createDataFrame(data, ["id", "name"])
display(df)

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("salary", when(col('id') == 1, 1000000)
                   .when (col('id') == 2, 200000)
                   .when (col('id') == 3, 300000)
                   .when (col('id') == 4, 400000).otherwise(500000))

# df = df.dropDuplicates()
display(df)

In [0]:
df_count = df.groupBy(*df.columns).count().filter("count > 1")
df_dropduplicates = df.dropDuplicates()
display(df_count)
display(df_dropduplicates)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *


df_3rd_hghest = df.withColumn("rnk", dense_rank().over(Window.orderBy(col("salary").desc())))
display(df_3rd_hghest.filter(col("rnk")==3))

In [0]:
df = df.withColumn("department", when((col("id") == 1) | (col("id") == 2), "IT")
                 .when((col("id") == 3) | (col("id") == 4), "HR")
                 .otherwise("Sales"))
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

df = df.withColumn("rank", dense_rank().over(Window.partitionBy("department").orderBy(col("salary").desc())))
df = df.filter(col("rank") == 1)
display(df)


In [0]:
data = [(1,"bharath",200000),(2,"arjun",100000),(3,"raj",300000),(4,"sai",20000),(1,"bharath",200000)]

df1 = spark.createDataFrame(data, ["id", "name","salary"])
display(df1)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df1 = df1.withColumn("rnk", row_number().over(Window.partitionBy("id").orderBy(col("salary").desc())))
display(df1.filter(col("rnk")> 1))

In [0]:
data = [(1,"bharath",200000),(2,"arjun",100000),(3,"raj",300000),(4,"sai",20000),(1,"bharath",200000)]

df2 = spark.createDataFrame(data, ["id", "name","salary"])
display(df2)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as _sum, col

window_spec = Window.orderBy("salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df2 = df2.withColumn("running_total", _sum("salary").over(window_spec))
display(df2)

In [0]:
data = [(1,"bharath",200000),(2,"arjun",100000),(3,"raj",300000),(4,"sai",20000),(1,"bharath",200000)]

df3 = spark.createDataFrame(data, ["id", "name","salary"])
display(df3)

In [0]:
df3.createOrReplaceTempView("table_test")

In [0]:
%sql
select *, sum(salary) over(order by salary asc rows between unbounded preceding and current row) as running_total from table_test